## Image Descriptions with Gemini 

Generate detailed textual descriptions for extracted images using Gemini 2.5 Flash.

**Prerequisites:**
- Make sure you rag-data dir with extracted dir like markdown, images and tables
- Google API key set in .env file

**Output:**
- Markdown descriptions saved to `data/rag-data/images_desc/{company}/{document}/page_X.md`

### Setup and Imports

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langfuse.langchain import CallbackHandler
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from PIL import Image
import base64
import io

### Configuration

In [ ]:
# Track
langfuse_handler = CallbackHandler()

# Paths
IMAGES_DIR = "data/rag-data-todo/images"
OUTPUT_DESC_DIR = "data/rag-data-todo/images_desc"

# Model configuration
MODEL_NAME = "gemini-2.5-flash-lite"
model = ChatGoogleGenerativeAI(model=MODEL_NAME)

# MODEL_NAME = "qwen3-vl"
# model = ChatOllama(model=MODEL_NAME, num_ctx=24576)

### Description Generation Function

In [20]:
describe_image_prompt = """Analyze this financial document page and extract meaningful data in a concise format.

For charts and graphs:
- Identify the metric being measured
- Must list key data points and values in details
- Note significant trends (growth, decline, stability)

For tables:
- Extract column headers and key rows in details
- Note important values and totals

For text:
- Summarize key facts and numbers only
- Skip formatting, headers, and navigation elements

Be direct and factual. Focus on numbers, trends, and insights that would be useful for retrieval."""

In [21]:
from langchain.messages import SystemMessage


def generate_image_description(image_path: Path):
    image = Image.open(image_path)
    buffered = io.BytesIO()
    image.save(buffered, format='PNG')

    image_base64 = base64.b64encode(buffered.getvalue()).decode()

    message = HumanMessage(
        content=[
            {'type': 'text', 'text': describe_image_prompt},
            {
                'type': 'image_url',
                'image_url': {'url': f"data:image/png;base64,{image_base64}"}
            }
        ]
    )
    system_prompt = SystemMessage('You are an AI Assistant')

    response = model.invoke([system_prompt, message], config={"callbacks": [langfuse_handler]})

    return response.content

In [31]:
image_path = Path(r'data\rag-data\images\meta\meta 10-k 2024\page_64.png')

response = generate_image_description(image_path)

Media upload error: Failed to upload media due to unexpected error. Queue item marked as done. Error: [Errno 11001] getaddrinfo failed


In [32]:
print(response)

**Revenue Worldwide (in $ millions)**

*   **Metric:** Revenue
*   **Key Data Points:**
    *   Dec 31, 2022: 32,165
    *   Mar 31, 2023: 28,645
    *   Jun 30, 2023: 31,999
    *   Sep 30, 2023: 34,146
    *   Dec 31, 2023: 40,111
    *   Mar 31, 2024: 36,455
    *   Jun 30, 2024: 39,071
    *   Sep 30, 2024: 40,589
    *   Dec 31, 2024: 46,783
*   **Trends:** Overall growth trend with fluctuations. Significant increase from Sep 2023 to Dec 2023, followed by a dip, then recovery and continued growth towards the end of 2024.

**Revenue US & Canada (in $ millions)**

*   **Metric:** Revenue
*   **Key Data Points:**
    *   Dec 31, 2022: 15,636
    *   Mar 31, 2023: 14,422
    *   Jun 30, 2023: 15,190
    *   Sep 30, 2023: 18,585
    *   Dec 31, 2023: 16,847
    *   Mar 31, 2024: 17,609
    *   Jun 30, 2024: 21,783
    *   Sep 30, 2024: 12,000 (Note: This value seems unusually low compared to surrounding points and may be a data entry error or represent a specific event. The chart visua

In [6]:

def generate_and_save_description(image_path: Path):
    company_name = image_path.parent.parent.name
    doc_name = image_path.parent.name

    output_dir = Path(OUTPUT_DESC_DIR)/company_name/doc_name
    output_dir.mkdir(parents=True, exist_ok=True)

    desc_file = output_dir / f"{image_path.stem}.md"

    if desc_file.exists():
        return False
    
    description = generate_image_description(image_path)
    desc_file.write_text(description, encoding='utf-8')
    
    return True

### Revenue Trends by User Geography (2022–2024)  
**Metric**: Revenue in $ millions (ad + non-ad), broken into geographies.  

#### Key Data & Trends:  
- **Worldwide**:  
  - Total revenue rose from $32,165M (Dec 2022) to $48,385M (Dec 2024).  
  - Growth driven by all regions; highest total in Dec 2024.  

- **US & Canada**:  
  - Revenue: $15,636M (Dec 2022) → $21,783M (Dec 2024).  
  - Growth rate: +18% vs 2023 (relative to Dec 31, 2023).  

- **Europe**:  
  - Revenue: $7,050M (Dec 2022) → $11,503M (Dec 2024).  
  - Growth rate: +26% vs 2023.  

- **Asia-Pacific**:  
  - Revenue: $6,050M (Dec 2022) → $9,245M (Dec 2024).  
  - Growth rate: +22% vs 2023.  

- **Rest of World**:  
  - Revenue: $3,429M (Dec 2022) → $5,854M (Dec 2024).  
  - Growth rate: +31% vs 2023 (fastest growth among regions).  

#### Key Insights:  
- Mature markets (**US & Canada, Europe**) generate higher revenue due to large ad/market size.  
- Lower-rate geographies (**Asia-Pacific, Rest of World**) show str

In [8]:
image_path = Path(r'data\rag-data\images\meta\meta 10-k 2024\page_64.png')

response = generate_and_save_description(image_path)

print(response)

Media upload error: Failed to upload media due to unexpected error. Queue item marked as done. Error: [Errno 11001] getaddrinfo failed


True


In [ ]:
from tqdm import tqdm

images_path = Path(IMAGES_DIR)
image_files = list(images_path.rglob("page_*.png"))

for image_path in tqdm(image_files):
    response = generate_and_save_description(image_path)


100%|██████████| 77/77 [00:00<00:00, 14648.10it/s]
